# Agentic Librarian POC — Detailed Analysis

This notebook explains the runtime workflow and key code paths for the Librarian Recommendation POC (`tools/run_gradio_poc.py`). It also evaluates whether the current POC is an "agentic" application, and lists concrete changes required to make it agentic (including a short roadmap and example agent loop).

## 1) High-level workflow (user-facing)

1. A user opens the Gradio UI served by `tools/run_gradio_poc.py`.
2. The UI shows two inputs: `Student ID` and `Query / Interests` and a `Submit` button.
3. On submit, the UI calls the backend endpoint `/predict` which invokes `gradio_query(student_id, query_text)`.
4. `gradio_query` delegates to `recommend_by_student()` to compute recommendations (embedding-based or TF-IDF fallback).
5. The backend returns a JSON list of recommended items (book_id, title, score, blurb) which the UI renders.
6. The user may click flags or copy blurbs; the POC currently has no persistent user or audit store beyond in-memory objects and optional CSV append hooks in companion tools.

Diagram (text):
User Browser -> Gradio Frontend -> Gradio Backend (gradio_query) -> recommend_by_student -> embeddings/index or TF-IDF -> Response -> Browser

## 2) Code walkthrough — file: `tools/run_gradio_poc.py`

We'll walk through the main blocks of the script and describe purpose, inputs, outputs, and relevant edge-cases.

### 2.1 Catalog & synthetic data
- `catalog` (pandas.DataFrame): small synthetic list of books, each row contains `book_id`, `title`, `author`, `description`.
- `borrow_history` (pandas.DataFrame): synthetic borrow events used to build student profiles.

### 2.2 Indexing: TF-IDF fallback
- `build_tfidf(docs)` builds a dense (numpy) TF-IDF matrix for the catalog text. It's pure-Python + NumPy — avoids heavy sklearn/SciPy dependencies.
- Outputs: `vectors` (N x V numpy float array), `vocab` (list of tokens).
- Edge cases: empty documents, unseen query tokens (handled by zero-vector logic).

### 2.3 Student profiles
- The script computes `student_vecs` by averaging the TF-IDF vectors of books in a student's training history.
- These vectors are used to recommend items that are similar (cosine similarity) to a student's history. Edge case: student with no history -> empty behavior (no recommendations)

### 2.4 Embedding-based retrieval (optional)
- The script attempts to import `sentence_transformers` and `faiss` and, if available, builds a semantic embedding index using `all-MiniLM-L6-v2`.
- If embeddings are built, the code uses FAISS's IndexFlatIP with normalized vectors (inner product == cosine similarity after normalization).
- If the embedding step raises any exception, it falls back to the TF-IDF path and logs the error.

### 2.5 Recommendation logic: `recommend_by_student(student_id, query_text, top_k=5)`
- Inputs: `student_id` (optional), `query_text` (optional).
- Behavior summary:
  - If embeddings are available:
    - If student has history, compute mean of their book embeddings and query FAISS for nearest neighbors.
    - Else if query_text provided, embed query and search FAISS.
  - Else fallback TF-IDF path: compute cosine similarity between the student vector (or query vector) and the TF-IDF matrix.
- Returns: list of tuples `(book_id, score, catalog_row)`.

Edge-cases considered: missing student history, zero-length vectors (norm==0), tokens not in vocab.

### 2.6 Blurb generation
- `generate_blurb(row)` is a simple formatting helper returning a short blurb string. In production this could be an LLM-generated human-friendly summary with safety checks and a length/QA pipeline.

## 3) Is this an agentic app? — Short answer

No, the current POC is not agentic. It is a synchronous request/response web app: user triggers a request and the backend computes and returns recommendations. There is no autonomous agent loop, no persistent goal state, no planners or action executors, and no feedback-driven policy that autonomously takes actions on behalf of users.

Agentic system characteristics (brief):
- Autonomous decision-making: agent acts without direct user request (on schedules or triggers).
- Goal representation: explicit objectives, utilities, or constraints guiding behavior over time.
- Action space & executors: ability to call external APIs/services, send messages, or change system state beyond replying to a request.
- Monitoring & recovery: state, logs, and mechanisms to detect and recover from failures or harmful behavior.

The POC lacks all of the above — it's a classic backend service, not an agent.

## 4) What would we need to change to make it agentic? — Concrete list

Below are practical modifications (ordered from low to higher risk/effort) to convert the POC into an agentic system for librarian tasks.

1) Persistent state & goals
   - Add a persistent datastore (Postgres / SQLite for demo) to store student profiles, actions taken, audit logs, and a 'goal' table for agents (e.g., "increase reading engagement by X%").

2) Agent scheduler & loop
   - Add a scheduler (RQ, APScheduler, or Celery Beat) that runs periodic agent jobs (e.g., nightly recommendations, overdue reminders) and an agent loop that evaluates goals and selects actions.

3) Planner / policy module
   - Implement a planner that given a goal and current state produces a short plan (sequence of actions) and a policy which chooses the next action. This can be a small rule-based or LLM-guided planner.

4) Action executors & sandboxing
   - Define a small set of safe actions (send_email, create_suggestion_list, flag_content, enroll_in_club) and implement action executors with strict rate limits, monitoring, and human-in-the-loop approvals for sensitive actions.

5) Safety, auditing & overrides
   - Add policy checks (e.g., OPA rules) and a manual approval queue for higher-risk actions. Log all decisions & rationale for inspection.

6) Evaluation & metrics
   - Add metric collection (CTR on suggestions, borrow-rate lift) and an offline simulation harness to test agent behavior before deployment.

7) Human-in-the-loop controls
   - An interface for librarians or admins to review candidate plans and approve/deny them (or require approval for certain sensitive action types).

8) Rate-limits & kill-switches
   - Implement per-agent and global rate limits and an emergency stop (killswitch) to halt autonomous actions quickly.

These changes move the app from a synchronous recommender to a system that can observe, plan, and act autonomously under constraints.

## 5) Minimal example: an agent loop (pseudo-code / small Python sketch)

Below is a small illustrative agent loop you can adopt. It is intentionally minimal and uses a rule-based planner for clarity.


In [ ]:
# Minimal agent loop sketch (run as a scheduled job)
import time
from typing import Dict, Any

# ...existing code... (catalog, indexing, recommend_by_student, generate_blurb)

GOALS = [
    {
        "id": "increase-engagement",
        "description": "Encourage inactive students to borrow 1 book per month",
        "metric": "monthly_borrows",
        "threshold": 1,
    }
]

def simple_planner(state: Dict[str, Any], goal: Dict[str, Any]):
    # Very small policy: find students with 0 borrows last month and propose a suggestion email
    actions = []
    for sid in state['students']:
        if state['students'][sid]['last_30_days_borrows'] == 0:
            recs = recommend_by_student(student_id=sid, top_k=3)
            actions.append({'actor': sid, 'action': 'recommend_email', 'payload': recs})
    return actions

def executor(action):
    # Only a demo: instead of sending email, log the action and write to an audit table
    print('EXECUTE', action['actor'], action['action'], [r[0] for r in action['payload']])
    # persist to audit log, record metrics, etc.

# Scheduler would call this periodically (e.g. daily/nightly)
def agent_job():
    state = {'students': {'S1': {'last_30_days_borrows': 0}, 'S2': {'last_30_days_borrows': 2}}}
    plan = simple_planner(state, GOALS[0])
    for a in plan:
        executor(a)

# Example run
if __name__ == '__main__':
    agent_job()

## 6) Roadmap & recommended milestones (90-day example)

- Week 0-2: Add a demo datastore (SQLite) and audit log table; refactor code to read/write student profiles and actions to the DB.
- Week 2-4: Add a scheduler (APScheduler/RQ) and wire a simple nightly agent job (the minimal agent_job sketch above).
- Week 4-8: Implement a planner and a small set of safe actions + an approvals queue for risky actions. Add basic metrics collection (borrows, CTR).
- Week 8-12: Add policy & safety checks (simple regex/heuristics; later OPA). Integration tests and simulated runs in a sandbox environment.
- Week 12+: Iterate on LLM-guided planning and policy enforcement, and add human-in-the-loop review UX for librarians.

## 7) Safety checklist (must-haves before enabling autonomy)
- Action whitelisting: only allow a small set of predefined actions for agents.
- Human approvals: require librarian approval for actions that affect user data or send messages to guardians/parents.
- Rate limits: prevent spamming users or repeatedly performing the same action.
- Auditing & explainability: record agent rationale (short text) and enable replay of decision traces.
- Kill-switch: admin endpoint to pause agent execution globally.
- Simulation testing: run agent against historical data to check for unwanted behavior before deploying.

## 8) Acceptance criteria to call the system agentic
- The system autonomously runs scheduled jobs that change system state or external resources (emails, enrollments) without a direct user request.
- Goals are represented explicitly and influence a planner/policy that selects actions.
- Safety controls and human oversight are in place.

### Next steps I can take for you:
- Implement the minimal agent loop as a scheduled job and persist audit logs (low risk).
- Add a demo SQLite schema and refactor the POC to use it for student profiles & audit logs.
- Create an admin UI to approve/deny agent actions (human-in-the-loop).

Which of these should I do next?